# Nemotron Submission Packaging — 2026-04-28

Purpose:
- keep the known-good submission packaging flow frozen
- produce a valid `submission.zip` for Kaggle
- separate submission generation from local prompt evaluation

Current project conventions:
- `P3_metric_aligned` is the best local prompt template from offline eval
- however, this notebook is submission-only and does not run prompt evaluation
- this notebook creates a compatible LoRA adapter shell for Nemotron-3-Nano-30B


In [2]:
import os
import site
import shutil
import zipfile

import kagglehub
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM

# -----------------------------
# 0) Paths
# -----------------------------
OUTPUT_DIR = "/kaggle/working"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "nemotron_lora_adapter")
ZIP_PATH = os.path.join(OUTPUT_DIR, "submission.zip")
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# This is the key missing piece from the failing notebook.
# It mirrors the earlier known-good environment setup idea.
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/"
if os.path.exists(cutlass_pkg_path):
    site.addsitedir(cutlass_pkg_path)
    print("Added CUTLASS path:", cutlass_pkg_path)
else:
    print("WARNING: CUTLASS path not found:", cutlass_pkg_path)

if os.path.exists(ADAPTER_DIR):
    shutil.rmtree(ADAPTER_DIR)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print("MODEL_PATH:", MODEL_PATH)
print("ADAPTER_DIR:", ADAPTER_DIR)

torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# -----------------------------
# 1) Load base model
# -----------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch_dtype,
    trust_remote_code=True,
    device_map="auto",
)

# -----------------------------
# 2) Attach LoRA shell
# -----------------------------
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.save_pretrained(ADAPTER_DIR)

print("Saved files:", os.listdir(ADAPTER_DIR))

required = {"adapter_config.json", "adapter_model.safetensors"}
present = set(os.listdir(ADAPTER_DIR))
missing = required - present
if missing:
    raise ValueError(f"Missing required adapter files: {missing}")

print("Adapter shell ready.")

# -----------------------------
# 3) Zip submission
# -----------------------------
required_files = ["adapter_config.json", "adapter_model.safetensors"]

for fname in required_files:
    full_path = os.path.join(ADAPTER_DIR, fname)
    if not os.path.exists(full_path):
        raise FileNotFoundError(f"Required file missing: {full_path}")

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        full_path = os.path.join(ADAPTER_DIR, fname)
        zf.write(full_path, arcname=fname)

print("Created:", ZIP_PATH)
print("Zip size (MB):", os.path.getsize(ZIP_PATH) / 1024 / 1024)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    print("Zip contents:", zf.namelist())

assert os.path.exists(ZIP_PATH), "submission.zip was not created"
print("Done.")

MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
ADAPTER_DIR: /kaggle/working/nemotron_lora_adapter


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

Saved files: ['adapter_model.safetensors', 'README.md', 'adapter_config.json']
Adapter shell ready.
Created: /kaggle/working/submission.zip
Zip size (MB): 4.0884809494018555
Zip contents: ['adapter_config.json', 'adapter_model.safetensors']
Done.


In [3]:
required_files = ['adapter_config.json', 'adapter_model.safetensors']

for fname in required_files:
    full_path = os.path.join(ADAPTER_DIR, fname)
    if not os.path.exists(full_path):
        raise FileNotFoundError(f'Required file missing: {full_path}')

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        full_path = os.path.join(ADAPTER_DIR, fname)
        zf.write(full_path, arcname=fname)

print('Created:', ZIP_PATH)
print('Zip size (MB):', os.path.getsize(ZIP_PATH) / 1024 / 1024)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    print('Zip contents:', zf.namelist())

assert os.path.exists(ZIP_PATH), 'submission.zip was not created'
print('Done. This notebook is submission-only.')


Created: /kaggle/working/submission.zip
Zip size (MB): 4.0884809494018555
Zip contents: ['adapter_config.json', 'adapter_model.safetensors']
Done. This notebook is submission-only.
